In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:/PROJECTS/4-1/nefroai/datasets/processed/CKD_NHANES_cleaned.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (6303, 18)
    age  gender           ethnicity   bmi  bp_systolic  bp_diastolic  \
0  43.0    Male  Non-Hispanic Asian  27.0        135.0          98.0   
1  66.0    Male  Non-Hispanic White  33.5        121.0          84.0   
2  44.0  Female      Other Hispanic  29.7        111.0          79.0   
3  34.0    Male    Mexican American  30.2        110.0          72.0   
4  68.0  Female  Non-Hispanic White  42.6        143.0          76.0   

   serum_creatinine  blood_urea_nitrogen  albumin_serum  phosphorus  \
0              0.80                 11.0            4.3         3.7   
1              0.79                 24.0            3.9         3.2   
2              0.64                 10.0            3.7         3.8   
3              0.82                 17.0            4.3         3.3   
4              0.76                 15.0            3.7         3.5   

   bicarbonate  calcium  uric_acid  urine_albumin  urine_creatinine  \
0         24.0      9.6        5.1         

In [4]:
df = df[df["age"] >= 18].copy()

print("Adult dataset shape:", df.shape)

Adult dataset shape: (5636, 18)


In [5]:
def calculate_egfr(row):

    scr = row["serum_creatinine"]
    age = row["age"]

    if row["gender"] == "Female":
        k = 0.7
        alpha = -0.241
        sex_factor = 1.012
    else:
        k = 0.9
        alpha = -0.302
        sex_factor = 1.0

    egfr = (
        142
        * (min(scr / k, 1) ** alpha)
        * (max(scr / k, 1) ** -1.2)
        * (0.9938 ** age)
        * sex_factor
    )

    return egfr


df["egfr_calculated"] = df.apply(calculate_egfr, axis=1)

In [6]:
print(
    df[
        [
            "age",
            "gender",
            "serum_creatinine",
            "egfr_calculated"
        ]
    ].head(10)
)

    age  gender  serum_creatinine  egfr_calculated
0  43.0    Male              0.80       112.614180
1  66.0    Male              0.79        97.975999
2  44.0  Female              0.64       111.687296
3  34.0    Male              0.82       118.212538
4  68.0  Female              0.76        85.298556
5  31.0  Female              0.80       100.959522
6  33.0  Female              0.55       124.044304
7  74.0  Female              0.59        94.512138
8  39.0    Male              0.99        99.375157
9  51.0    Male              0.84       105.581019


In [7]:
print(df["egfr_calculated"].describe())

count    5636.000000
mean       92.239767
std        21.849150
min         3.508700
25%        78.350608
50%        94.261765
75%       107.406718
max       148.977312
Name: egfr_calculated, dtype: float64


In [8]:
df["ckd_reference"] = (
    (df["egfr_calculated"] < 60) |
    (df["albumin_creatinine_ratio"] >= 30)
).astype(int)

In [9]:
print(df["ckd_reference"].value_counts())

print(
    df["ckd_reference"]
    .value_counts(normalize=True)
    * 100
)

ckd_reference
0    4669
1     967
Name: count, dtype: int64
ckd_reference
0    82.842441
1    17.157559
Name: proportion, dtype: float64


In [10]:
print(
    pd.crosstab(
        df["ckd_present"],
        df["ckd_reference"],
        margins=True
    )
)

ckd_reference     0    1   All
ckd_present                   
0              3000    0  3000
1              1669  967  2636
All            4669  967  5636


In [11]:
selected_features = [
    "age",
    "gender",
    "bp_systolic",
    "bp_diastolic",
    "serum_creatinine",
    "albumin_creatinine_ratio",
    "diabetes_diagnosed"
]

X = df[selected_features]
y = df["ckd_reference"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5636, 7)
y shape: (5636,)


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (4508, 7)
Testing: (1128, 7)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["gender"]

numeric_features = [
    "age",
    "bp_systolic",
    "bp_diastolic",
    "serum_creatinine",
    "albumin_creatinine_ratio",
    "diabetes_diagnosed"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(4508, 8)
(1128, 8)


In [14]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train_processed, y_train)

GradientBoostingClassifier(learning_rate=0.05, n_estimators=200,
                           random_state=42)

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test_processed)
y_prob = model.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.9955673758865248
Precision: 0.9846153846153847
Recall   : 0.9896907216494846
F1 Score : 0.9871465295629821
ROC-AUC  : 0.9999061789443475

Confusion Matrix:
[[931   3]
 [  2 192]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       934
           1       0.98      0.99      0.99       194

    accuracy                           1.00      1128
   macro avg       0.99      0.99      0.99      1128
weighted avg       1.00      1.00      1.00      1128



In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train_processed, y_train)

rf_pred = rf_model.predict(X_test_processed)
rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

print("Random Forest")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test, rf_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest
Accuracy : 0.9946808510638298
Precision: 0.9895833333333334
Recall   : 0.979381443298969
F1 Score : 0.9844559585492227
ROC-AUC  : 0.999900660058721

Confusion Matrix:
[[932   2]
 [  4 190]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       934
           1       0.99      0.98      0.98       194

    accuracy                           0.99      1128
   macro avg       0.99      0.99      0.99      1128
weighted avg       0.99      0.99      0.99      1128



In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)

svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)
svm_prob = svm_model.predict_proba(X_test_scaled)[:, 1]

print("SVM")
print("Accuracy :", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall   :", recall_score(y_test, svm_pred))
print("F1 Score :", f1_score(y_test, svm_pred))
print("ROC-AUC  :", roc_auc_score(y_test, svm_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

print("\nClassification Report:")
print(classification_report(y_test, svm_pred))

SVM
Accuracy : 0.9166666666666666
Precision: 0.9807692307692307
Recall   : 0.5257731958762887
F1 Score : 0.6845637583892616
ROC-AUC  : 0.9702366498156694

Confusion Matrix:
[[932   2]
 [ 92 102]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95       934
           1       0.98      0.53      0.68       194

    accuracy                           0.92      1128
   macro avg       0.95      0.76      0.82      1128
weighted avg       0.92      0.92      0.91      1128



In [20]:
import joblib

joblib.dump(
    model,
    "gradient_boosting_rural_corrected_model.pkl"
)

joblib.dump(
    preprocessor,
    "rural_corrected_preprocessor.pkl"
)

['rural_corrected_preprocessor.pkl']